> Projeto Desenvolve <br>
Programação Intermediária com Python <br>
Profa. Camila Laranjeira (mila@projetodesenvolve.com.br) <br>

# 3.14 - ORM

## Exercícios

#### Q1. Conhecendo os dados
Baixe o seguinte csv onde iremos trabalhar. Ele contém informações sobre salários de profissionais de dados de uma empresa hipotética entre 2009 e 2016
* https://github.com/camilalaranjeira/python-intermediario/blob/main/salaries.csv

Suas colunas, descritas na [página do Kaggle que contém o dataset](https://www.kaggle.com/datasets/krishujeniya/salary-prediction-of-data-professions?resource=download), são:
* FIRST NAME: Primeiro nome do profissional de dados (String)
* LAST NAME: Sobrenome do profissional de dados (String)
* SEX: Gênero do profissional de dados (String: 'F' para Feminino, 'M' para Masculino)
* DOJ (Date of Joining): A data em que o profissional de dados ingressou na empresa (Data no formato MM/DD/AAAA)
* CURRENT DATE: A data atual ou a data de referência dos dados (Data no formato MM/DD/AAAA)
* DESIGNATION: O cargo ou designação do profissional de dados (String: ex., Analista, Analista Sênior, Gerente)
* AGE: Idade do profissional de dados (Integer)
* SALARY: Salário anual do profissional de dados (Float)
* UNIT: Unidade de negócios ou departamento em que o profissional de dados trabalha (String: ex., TI, Finanças, Marketing)
* LEAVES USED: Número de licenças utilizadas pelo profissional de dados (Integer)
* LEAVES REMAINING: Número de licenças restantes para o profissional de dados (Integer)
* RATINGS: Avaliações de desempenho do profissional de dados (Float)
* PAST EXP: Experiência de trabalho anterior em anos antes de ingressar na empresa atual (Float)

Na célula a seguir, **carregue os dados do CSV e dê uma olhada neles antes de seguir**.

In [ ]:
### Escreva sua resposta aqui

import pandas as pd

url = "https://raw.githubusercontent.com/camilalaranjeira/python-intermediario/main/salaries.csv"
df = pd.read_csv(url)

df.head()

#### Q2. Modelando os dados
Você deve **criar um ORM com SQLAlchemy capaz de comportar os dados dessa base**.

* Crie um campo de chave primária `ID`, que deve ser incrementado automaticamente
* Os campos SEX, DESIGNATION e UNIT devem ser definidos como classes `Enum` com os possíveis valores (consulte os valores únicos dessas colunas)
* Para os outros campos, consulte os tipos de dados informados na descrição acima

In [ ]:
### Escreva sua resposta aqui

from sqlalchemy import Column, Integer, String, Float, Enum
from sqlalchemy.orm import declarative_base
import enum

Base = declarative_base()


class SexEnum(enum.Enum):
    Male = "Male"
    Female = "Female"


class Salary(Base):
    __tablename__ = "salaries"

    id = Column(Integer, primary_key=True, autoincrement=True)
    first_name = Column(String)
    last_name = Column(String)
    sex = Column(Enum(SexEnum))
    designation = Column(String)
    unit = Column(String)
    salary = Column(Float)
    leave_balance = Column(Integer)

#### Q3. Estabelecendo uma conexão

Usando o método `create_engine` do SQLAlchemy, crie uma conexão com um novo banco de dados SQLite chamado `salarios`.

In [ ]:
### Escreva sua resposta aqui

from sqlalchemy import create_engine

engine = create_engine("sqlite:///salarios.db")

#### Q4. Criando as tabelas
Crie as tabelas da questão Q2 no banco `salarios`.

In [ ]:
### Escreva sua resposta aqui

Base.metadata.create_all(engine)

#### Q5. Populando

Usando o método `to_sql` da biblioteca Pandas (veja [a documentação](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_sql.html)), popule o banco `salarios` com os dados do csv que você carregou na questão Q1.
* Lembre-se de definir o parâmetro `if_exists='append'` para que as tabelas não sejam dropadas e recriadas.

In [ ]:
### Escreva sua resposta aqui

df.to_sql("salaries", engine, if_exists="append", index=False)

#### Q6. Consultas SQL vs ORM

Agrupe os dados por DESIGNATION e selecione o mínimo, máximo e a média dos salários (SALARY) divididos por 12. Já que o atributo SALARY é anual, dividir por 12 nos mostrará os valores mensais.

Assumindo que a variável que armazena a sua conexão se chama `engine`, você deve realizar a query acima de três formas:
* Executando a query SQL através de uma instância de conexão retornada pelo método `engine.connect()`
* Executando a query SQL com o método `read_sql_query` do Pandas (veja [a documentação](https://pandas.pydata.org/docs/reference/api/pandas.read_sql_query.html)). Você usará mesma instância `engine.connect()` como um dos parâmetros.
* Executando uma query criada com o módulo `select` do SQLAlchemy. Sua execução deve ser feita através de um objeto `Session` do módulo `orm` do SQLAlchemy (`Session(engine)`).


In [ ]:
### Execute aqui sua query SQL com SQLAlchemy

with engine.connect() as conn:
    result = conn.execute("""
    SELECT designation,
           MIN(salary)/12 as min_salary,
           MAX(salary)/12 as max_salary,
           AVG(salary)/12 as avg_salary
    FROM salaries
    GROUP BY designation
    """)
    
    for row in result:
        print(row)

In [ ]:
### Execute aqui sua query SQL com SQLAlchemy + Pandas

query = """
SELECT designation,
       MIN(salary)/12 as min_salary,
       MAX(salary)/12 as max_salary,
       AVG(salary)/12 as avg_salary
FROM salaries
GROUP BY designation
"""

pd.read_sql_query(query, engine)

In [ ]:
### Execute aqui sua query com SQLAlchemy ORM

from sqlalchemy.orm import Session
from sqlalchemy import func

session = Session(engine)

result = session.query(
    Salary.designation,
    func.min(Salary.salary)/12,
    func.max(Salary.salary)/12,
    func.avg(Salary.salary)/12
).group_by(Salary.designation).all()

for r in result:
    print(r)